## EDA

### 환경 설정

In [1]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re
from tqdm import tqdm
from wordcloud import WordCloud
from collections import Counter
from kiwipiepy import Kiwi
import os

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 텍스트 분석

### 사용자 사전 구축 및 형태소 분석

In [2]:
# 데이터 로드
df_master = pd.read_csv('./final_data/master_preprocessed_260430.csv')
df = df_master.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1170945 entries, 0 to 1170944
Data columns (total 31 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   collect_date           1170945 non-null  str    
 1   platform               1170945 non-null  str    
 2   review_id              1170945 non-null  float64
 3   product_id             1170945 non-null  object 
 4   product_name           1170945 non-null  str    
 5   review_date            1170945 non-null  str    
 6   year                   1170945 non-null  int64  
 7   month                  1170945 non-null  int64  
 8   content                1170945 non-null  str    
 9   rating                 1170945 non-null  int64  
 10  helpful_count          1170945 non-null  int64  
 11  has_image              1170945 non-null  int64  
 12  purchase_option        1170945 non-null  str    
 13  purchase_option_color  1170945 non-null  str    
 14  purchase_option_size   117094

In [3]:
# 브랜드별 마스터 데이터 로드
df_andar = pd.read_csv('./final_data/안다르_master.csv')
df_fila = pd.read_csv('./final_data/FILA_master_v2.csv')
df_xexymix = pd.read_csv('./final_data/젝시믹스_master.csv')
df_lululemon = pd.read_csv('./final_data/룰루레몬_master.csv')

In [4]:
def clean_text(text):
    # 1. 결측치 처리
    if pd.isna(text):
        return ' '

    text = str(text)

    # 2. HTML 태그 제거
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. URL 제거
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # 4. 이메일 제거
    text = re.sub(r'\S+@\S+', ' ', text)

    # 5. 이모지 & 이모티콘 제거
    text = re.sub(r'[^\w\s\uAC00-\uD7A3\u3130-\u318F]', ' ', text)
    # \uAC00-\uD7A3 : 한글 완성형
    # \u3130-\u318F : ㅎ, ㅋ 같은 한글 자모음 (반복 제거 대상이라 유지)

    # 6. 줄바꿈, 탭 → 공백으로 변환
    text = re.sub(r'[\n\r\t]', ' ', text)

    # 7. 반복 문자 1개로 통일 (ㅎㅎㅎ→ㅎ, ㅋㅋㅋ→ㅋ, ~~~→~)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 8. 연속 공백 → 공백 1개로 통일
    text = re.sub(r'\s+', ' ', text)

    # 9. 대소문자 통합 (영문 소문자)
    text = text.lower()

    # 10. 앞뒤 공백 제거
    text = text.strip()

    return text

df_andar['content'] = df_andar['content'].apply(clean_text)
df_fila['content'] = df_fila['content'].apply(clean_text)
df_xexymix['content'] = df_xexymix['content'].apply(clean_text)
df_lululemon['content'] = df_lululemon['content'].apply(clean_text)

In [5]:
df_andar.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
0,2026-04-21,자사몰,48724074.0,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-14,2026,4,러닝 후 몸 풀어주는 용으로 아주 좋아요 종아리나 등 목 뒤에 사용하기 좋아요 색깔...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,155~159cm,45~49kg,https://andar.co.kr/product/detail.html?produc...,안다르,용품,기타,필라테스/요가,여성,9900.0,9900.0,10.0,False,False
1,2026-04-21,자사몰,48590531.0,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-07,2026,4,딱딱하네요 저는 말랑말랑 한줄 알았어요 2026 04 07 20 15 59 에 등록...,3,0,0,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,https://andar.co.kr/product/detail.html?produc...,안다르,용품,기타,필라테스/요가,여성,9900.0,9900.0,10.0,False,False
2,2026-04-21,자사몰,48535840.0,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,요가링이랑 폼롤러랑 같이 사서 집에서 잘 굴려주고 있어요 색깔도 너무 예쁘고 귀여워...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,165~169cm,55~59kg,https://andar.co.kr/product/detail.html?produc...,안다르,용품,기타,필라테스/요가,여성,9900.0,9900.0,10.0,False,False
3,2026-04-21,자사몰,48535469.0,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,묵직하니 사용하기 편하네요 릴렉스 마사지 듀얼볼 ⅱ 후기 정사이즈,5,0,0,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,160~164cm,50~54kg,https://andar.co.kr/product/detail.html?produc...,안다르,용품,기타,필라테스/요가,여성,9900.0,9900.0,10.0,False,False
4,2026-04-21,자사몰,48521693.0,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-03,2026,4,시원하게 마사지가되요 여기저기 쉽게 문지르고 아픈 부위 꾹꾹 눌러주니 시원하네요 릴...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,160~164cm,55~59kg,https://andar.co.kr/product/detail.html?produc...,안다르,용품,기타,필라테스/요가,여성,9900.0,9900.0,10.0,False,False


In [6]:
df_fila.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
0,2026-04-28,자사몰,139954.0,8127678283812,휠라 스피드서브 T9 2.0,2026-03-26,2026,3,이 신발 네트 대시가 왜 이렇게 빨라 스피드서브 t9 2 0 실착리뷰 테니스 구력은...,5,28,1,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,https://www.fila.co.kr/products/1100fs261tn02x...,FILA,신발,운동화,테니스,여성,159000.0,159000.0,1.0,1.0,1.0
1,2026-04-28,자사몰,143752.0,8100165845028,하레핀 1998,2026-04-21,2026,4,후기 사진은 크게 확인 부탁드립니다 남자친구랑 사귀고 첫 커플 신발인데 뭐를 살까 ...,5,25,1,270,unknown,270,0,NaN,NaN,unknown,unknown,https://www.fila.co.kr/products/1100fs261od03x...,FILA,신발,스니커즈,일반,여성,139000.0,139000.0,2.0,1.0,1.0
2,2026-04-28,자사몰,140502.0,8127678283812,휠라 스피드서브 T9 2.0,2026-03-31,2026,3,엑실러스3 유저의 스피드서브 2 0 기변 후기 테니스 구력 2년 차 그동안 저의 베...,5,22,1,240,unknown,240,0,NaN,NaN,unknown,unknown,https://www.fila.co.kr/products/1100fs261tn02x...,FILA,신발,운동화,테니스,여성,159000.0,159000.0,1.0,1.0,1.0
3,2026-04-28,자사몰,143962.0,8100165845028,하레핀 1998,2026-04-24,2026,4,첫 커플신발로 휠라 하레핀1998을 선택했어요 일반적인 스니커즈 디자인이 질려서 고...,5,22,1,280,unknown,280,0,NaN,NaN,unknown,unknown,https://www.fila.co.kr/products/1100fs261od03x...,FILA,신발,스니커즈,일반,여성,139000.0,139000.0,2.0,1.0,1.0
4,2026-04-28,자사몰,140489.0,8127678283812,휠라 스피드서브 T9 2.0,2026-03-30,2026,3,스피드서브 1 찐유저의 2 0 버전업 후기 단종의 눈물을 닦아준 완벽한 진화 fea...,5,20,1,265,unknown,265,0,NaN,NaN,unknown,unknown,https://www.fila.co.kr/products/1100fs261tn02x...,FILA,신발,운동화,테니스,여성,159000.0,159000.0,1.0,1.0,1.0


In [7]:
df_lululemon.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
0,2026-04-17,자사몰,361263497.0,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,2025-10-05,2025,10,골프복 상의로 아주 좋습니다 재질도 좋고 핏도 아주 좋아요 활동성이 좋아서 골프복으...,5,0,0,L,unknown,L,0,NaN,NaN,unknown,75-79kg,NaN,룰루레몬,상의,반팔티셔츠,골프,Men,138000.0,138000.0,10.0,True,False
1,2026-04-17,자사몰,354639167.0,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,2025-08-19,2025,8,기능이 좋습니다 상품이 너무 편하고 착용감이 좋아요 땀이 많이 나고 자국도 안생기고...,5,0,0,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,NaN,룰루레몬,상의,반팔티셔츠,골프,Men,138000.0,138000.0,10.0,True,False
2,2026-04-17,자사몰,354121464.0,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,2025-08-12,2025,8,디자인도 깔끔하고 편합니다 디자인이 깔끔하여 한달전에 구매해서 편하고 가벼워서 계속...,5,0,0,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,NaN,룰루레몬,상의,반팔티셔츠,골프,Men,138000.0,138000.0,10.0,True,False
3,2026-04-17,자사몰,353645777.0,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,2025-08-07,2025,8,원단이 좋습니다 옷이 너무 디자인이 깔끔하고 원단이 좋습니다 나중에 다른색으로 하나...,5,0,0,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,NaN,룰루레몬,상의,반팔티셔츠,골프,Men,138000.0,138000.0,10.0,True,False
4,2026-04-17,자사몰,349934726.0,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,2025-06-17,2025,6,세련된 색감 쾌적한 착용감 엣지있는 디자인 착용감이 편안하고 시원해요 핏이 박시하지...,4,0,0,unknown,unknown,unknown,0,NaN,NaN,unknown,unknown,NaN,룰루레몬,상의,반팔티셔츠,골프,Men,138000.0,138000.0,10.0,True,False


In [8]:
df_xexymix.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
0,2026-04-20,자사몰,5032253.0,2070608,UV 쉴드 에어핏 페이스 마스크,2026-03-09,2026,3,친구 소개로 구매했어요 친구가 걷기운동할때 필요해서 이것저것 구매했는데 기장 편하게...,5,1,1,색상:화이트 / 사이즈:L,화이트,L,0,158.0,50.0,155~159cm,50~54kg,https://www.xexymix.com/shop/shopdetail.html?b...,젝시믹스,용품,기타,골프,기타,19000.0,19000.0,3.0,False,True
1,2026-04-20,자사몰,4998316.0,2070608,UV 쉴드 에어핏 페이스 마스크,2026-01-26,2026,1,l사이즈 사고 원단도 좋고 운동시 호흡도 편한데 큰 느낌이라 작은사이즈 다시 주문했...,5,0,1,색상:블랙 / 사이즈:S/M,블랙,S/M,0,168.0,55.0,165~169cm,55~59kg,https://www.xexymix.com/shop/shopdetail.html?b...,젝시믹스,용품,기타,골프,기타,19000.0,19000.0,3.0,False,True
2,2026-04-20,자사몰,4936161.0,2070608,UV 쉴드 에어핏 페이스 마스크,2025-12-14,2025,12,구매하여 처음 사용해봤는데 가성비 진짜 좋네요 바람도 잘 막아주고 추운날씨에도 딱 ...,5,0,1,색상:블랙 / 사이즈:L,블랙,L,0,179.0,70.0,180~184cm,70~74kg,https://www.xexymix.com/shop/shopdetail.html?b...,젝시믹스,용품,기타,골프,기타,19000.0,19000.0,3.0,False,True
3,2026-04-20,자사몰,4727267.0,2070608,UV 쉴드 에어핏 페이스 마스크,2025-07-22,2025,7,전부터 풀페이스마스크 하나 사려고 보고 있다가 지난번 블랙색상 사고 마음에 들어서 ...,5,4,1,색상:화이트 / 사이즈:L,화이트,L,0,170.0,60.0,170~174cm,60~64kg,https://www.xexymix.com/shop/shopdetail.html?b...,젝시믹스,용품,기타,골프,기타,19000.0,19000.0,3.0,False,True
4,2026-04-20,자사몰,4821424.0,2070608,UV 쉴드 에어핏 페이스 마스크,2025-10-10,2025,10,낮에 러닝하는데 얼굴 타는게 신경쓰였는데 마스크 답답하지 않고 흘러 내리지 않아 좋아요,5,1,1,색상:화이트 / 사이즈:S/M,화이트,S/M,0,153.0,45.0,150~154cm,45~49kg,https://www.xexymix.com/shop/shopdetail.html?b...,젝시믹스,용품,기타,골프,기타,19000.0,19000.0,3.0,False,True


### 정식 kiwi 형태소 분석 전 브랜드별 워크클라우드 시각화 진행

##### 사용자사전, 불용어 리스트만 수정해서 사용

##### 1차 워드클라우드 진행

In [9]:
# # kiwi 기초 설정 및 브랜드별 워드클라우드 진행
# from collections import Counter

# # Kiwi 초기화
# kiwi = Kiwi()

# # 1. 최소한의 사용자 사전 추가 (브랜드명 보호)
# brands_list = ['안다르', '휠라', '젝시믹스', '룰루레몬']
# for b in brands_list:
#     kiwi.add_user_word(b, 'NNP')

# # 2. 토큰 추출 함수 정의
# def get_tokens(text):
#     if not text or len(text.strip()) == 0:
#         return ""
    
#     result = kiwi.tokenize(text)
#     tokens = []
    
#     for t in result:
#         # 명사 및 어근(쫀쫀, 탄탄 등 핵심 표현) 추출
#         if t.tag in ['NNG', 'NNP', 'XR']:
#             if len(t.form) > 1:
#                 tokens.append(t.form)
#         # 형용사 추출 (가공: 예쁘 -> 예쁘다)
#         elif t.tag == 'VA':
#             tokens.append(t.form + '다')
            
#     return " ".join(tokens)

# # 3. 각 데이터프레임에 적용 (새 컬럼 생성)
# df_andar['tokens'] = df_andar['content'].apply(get_tokens)
# df_fila['tokens'] = df_fila['content'].apply(get_tokens)
# df_xexymix['tokens'] = df_xexymix['content'].apply(get_tokens)
# df_lululemon['tokens'] = df_lululemon['content'].apply(get_tokens)

In [10]:
# import matplotlib.pyplot as plt
# from wordcloud import WordCloud

# # 시각화할 데이터프레임과 이름 매핑
# dfs = [
#     (df_andar, 'Andar'),
#     (df_fila, 'FILA'),
#     (df_xexymix, 'Xexymix'),
#     (df_lululemon, 'Lululemon')
# ]

# # 공통 불용어 설정 (결과를 보고 계속 추가하세요)
# stopwords = {'진짜', '너무', '구매', '생각', '정도', '같아요', '있어요', '많이', '조금', '그냥'}

# for df, name in dfs:
#     # 해당 브랜드의 모든 토큰 합치기
#     all_text = " ".join(df['tokens'])
#     word_list = all_text.split()
    
#     # 불용어 제거 및 빈도 계산
#     filtered_words = [w for w in word_list if w not in stopwords]
#     counts = Counter(filtered_words)
    
#     # 워드클라우드 생성 (Mac 전용 폰트 경로)
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=800,
#         height=400,
#         max_words=100
#     ).generate_from_frequencies(counts)
    
#     # 시각화 출력
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Word Cloud - {name}", fontsize=20)
#     plt.axis('off')
#     plt.show()

In [11]:
# import pandas as pd
# from kiwipiepy import Kiwi
# from tqdm import tqdm
# from wordcloud import WordCloud
# from collections import Counter
# import matplotlib.pyplot as plt

# # 1. 데이터 샘플링 (117만 건 전체 실행 전 1만 건만 테스트)
# # 테스트가 성공적으로 끝나면 아래 줄에서 '.sample(n=10000, random_state=42)' 부분을 지우면 됩니다.
# df_list = [df_andar, df_fila, df_xexymix, df_lululemon]
# combined_df = pd.concat([df[['brand', 'content']].copy() for df in df_list], ignore_index=True)
# test_df = combined_df.sample(n=10000, random_state=42).copy()

# # 2. Jupyter 충돌을 피하기 위한 기본 설정
# kiwi = Kiwi()

# user_words = ['안다르', '휠라', '젝시믹스', '룰루레몬', '에어리핏', '에어스트', '에어쿨링', '기모', '조거팬츠']
# for word in user_words:
#     kiwi.add_user_word(word, 'NNP')

# def extract_keywords(text):
#     if not isinstance(text, str) or not text.strip(): return ""
#     try:
#         tokens = kiwi.tokenize(text)
#         return " ".join([
#             t.form + '다' if t.tag == 'VA' else t.form
#             for t in tokens
#             if (t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form == '핏')) or t.tag == 'VA'
#         ])
#     except Exception:
#         return ""

# print("순차적 형태소 분석 시작 (샘플 데이터)...")

# # 3. .apply()를 활용한 안전한 직렬 처리
# tqdm.pandas(desc="형태소 분석")
# test_df['tokens'] = test_df['content'].progress_apply(extract_keywords)

# print("분석 완료! 시각화를 준비합니다.\n")

# # 4. 시각화 셀 설정
# # 하드코딩 대신 데이터에 존재하는 브랜드만 동적 추출
# brand_names = test_df['brand'].unique()

# refined_stopwords = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '구매', '제품', '상품', '주문', '배송', 
#     '이번', '선택', '사용', '아울렛', '로그인', '사이트', '재구매', '이용', '하나', '정도',
#     '좋다', '같다', '있다', '없다', '크다', '작다', '적당하다', '괜찮다', '그냥', '진짜', 
#     '너무', '생각', '느낌', '부분', '사람', '마음', '소리', '보고', '조금', '가장',
#     '사이즈', '색상', '컬러', '화면', '실물', '사진', '재질', '소재', '다르다', '정말', 
#     '역시', '보통', '원래', '의사', '추천', '다시', '자주', '한번', '어디', '확인'
# }

# for b_name in brand_names:
#     brand_df = test_df[test_df['brand'] == b_name]
#     if brand_df.empty: continue

#     all_words = " ".join(brand_df['tokens']).split()
#     filtered_words = [w for w in all_words if w not in refined_stopwords]
    
#     if not filtered_words: continue # 필터링 후 단어가 없으면 시각화 건너뜀
        
#     counts = Counter(filtered_words)
    
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=1000, height=600, max_words=60, colormap='Dark2'
#     ).generate_from_frequencies(counts)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Brand : {b_name} (Sample)", fontsize=20, fontweight='bold')
#     plt.axis('off')
#     plt.show()

##### 2차 워드클라우드 진행

In [12]:
# import pandas as pd
# from kiwipiepy import Kiwi
# from tqdm import tqdm
# from wordcloud import WordCloud
# from collections import Counter
# import matplotlib.pyplot as plt

# # 1. 데이터 샘플링 (117만 건 전체 실행 전 1만 건만 테스트)
# # 테스트가 성공적으로 끝나면 아래 줄에서 '.sample(n=10000, random_state=42)' 부분을 지우면 됩니다.
# df_list = [df_andar, df_fila, df_xexymix, df_lululemon]
# combined_df = pd.concat([df[['brand', 'content']].copy() for df in df_list], ignore_index=True)
# test_df = combined_df.copy()  # 샘플링 제거 후 전체 데이터로 분석할 때는 .copy()만 사용

# # 2. Jupyter 충돌을 피하기 위한 기본 설정
# kiwi = Kiwi()

# user_words = ['안다르', '휠라', '젝시믹스', '룰루레몬', '에어리핏', '에어스트', '에어쿨링', '기모', '조거팬츠']
# for word in user_words:
#     kiwi.add_user_word(word, 'NNP')

# def extract_keywords(text):
#     if not isinstance(text, str) or not text.strip(): return ""
#     try:
#         tokens = kiwi.tokenize(text)
#         return " ".join([
#             t.form + '다' if t.tag == 'VA' else t.form
#             for t in tokens
#             if (t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form == '핏')) or t.tag == 'VA'
#         ])
#     except Exception:
#         return ""

# print("순차적 형태소 분석 시작 (샘플 데이터)...")

# # 3. .apply()를 활용한 안전한 직렬 처리
# tqdm.pandas(desc="형태소 분석")
# test_df['tokens'] = test_df['content'].progress_apply(extract_keywords)

# print("분석 완료! 시각화를 준비합니다.\n")

# # 4. 시각화 셀 설정
# # 하드코딩 대신 데이터에 존재하는 브랜드만 동적 추출
# brand_names = test_df['brand'].unique()

# refined_stopwords = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '구매', '제품', '상품', '주문', '배송', 
#     '이번', '선택', '사용', '아울렛', '로그인', '사이트', '재구매', '이용', '하나', '정도',
#     '좋다', '같다', '있다', '없다', '크다', '작다', '적당하다', '괜찮다', '그냥', '진짜', 
#     '너무', '생각', '느낌', '부분', '사람', '마음', '소리', '보고', '조금', '가장',
#     '사이즈', '색상', '컬러', '화면', '실물', '사진', '재질', '소재', '다르다', '정말', 
#     '역시', '보통', '원래', '의사', '추천', '다시', '자주', '한번', '어디', '확인'
# }

# for b_name in brand_names:
#     brand_df = test_df[test_df['brand'] == b_name]
#     if brand_df.empty: continue

#     all_words = " ".join(brand_df['tokens']).split()
#     filtered_words = [w for w in all_words if w not in refined_stopwords]
    
#     if not filtered_words: continue # 필터링 후 단어가 없으면 시각화 건너뜀
        
#     counts = Counter(filtered_words)
    
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=1000, height=600, max_words=60, colormap='Dark2'
#     ).generate_from_frequencies(counts)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Brand : {b_name} (Sample)", fontsize=20, fontweight='bold')
#     plt.axis('off')
#     plt.show()

3차 워드클라우드 진행

In [13]:
# import os

# # 1. 데이터 합치기
# df_list = [df_andar, df_fila, df_xexymix, df_lululemon]
# combined_df = pd.concat([df[['brand', 'content']].copy() for df in df_list], ignore_index=True)
# test_df = combined_df.copy()

# # 2. Kiwi 초기화 (num_workers=os.cpu_count() → C++ 워커 스레드 활성화, Jupyter 충돌 없음)
# kiwi = Kiwi(num_workers=os.cpu_count())

# user_words = [
#     '안다르', '휠라', '젝시믹스', '룰루레몬', 
#     '에어리핏', '에어스트', '에어쿨링', '기모', '조거팬츠',
#     '얼라인', '스쿠바', '스위프틀리', '맨즈', '롱슬리브', '숏슬리브',
#     '블랙라벨', '에샤페', '테니스'
# ]
# for word in user_words:
#     kiwi.add_user_word(word, 'NNP')

# def process_result(tokens):
#     return " ".join([
#         t.form + '다' if t.tag == 'VA' else t.form
#         for t in tokens
#         if (t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form == '핏')) or t.tag == 'VA'
#     ])

# print("형태소 분석 시작 (C++ 멀티코어 배치 처리)...")

# # 3. 리스트 일괄 전달 → kiwipiepy 내부 C++ 병렬 배치 처리
# contents = test_df['content'].fillna('').astype(str).tolist()
# test_df['tokens'] = [process_result(r) for r in kiwi.tokenize(contents)]

# print("분석 완료! 시각화를 준비합니다.\n")

# # 4. 시각화
# brand_names = test_df['brand'].unique()

# refined_stopwords = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '구매', '제품', '상품', '주문', '배송', 
#     '이번', '선택', '사용', '아울렛', '로그인', '사이트', '재구매', '이용', '하나', '정도',
#     '좋다', '같다', '있다', '없다', '크다', '작다', '적당하다', '괜찮다', '그냥', '진짜', 
#     '너무', '생각', '느낌', '부분', '사람', '마음', '소리', '보고', '조금', '가장',
#     '사이즈', '색상', '컬러', '화면', '실물', '사진', '재질', '소재', '다르다', '정말',
#     '편하다', '예쁘다', '이쁘다', '편안하다', '만족', '최고', '불편',
#     '평소', '처음', '요즘', '항상', '자주',
#     '고민', '구입', '선물', '남편', '추가', '입다', '신다', '많다', '길이'
# }

# for b_name in brand_names:
#     brand_df = test_df[test_df['brand'] == b_name]
#     if brand_df.empty: continue

#     all_words = " ".join(brand_df['tokens']).split()
#     filtered_words = [w for w in all_words if w not in refined_stopwords]
    
#     if not filtered_words: continue
        
#     counts = Counter(filtered_words)
    
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=1000, height=600, max_words=60, colormap='Dark2'
#     ).generate_from_frequencies(counts)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Brand : {b_name}", fontsize=20, fontweight='bold')
#     plt.axis('off')
#     plt.show()

##### 4차 워드클라우드 진행

In [14]:
# # 1. 데이터 합치기
# df_list = [df_andar, df_fila, df_xexymix, df_lululemon]
# combined_df = pd.concat([df[['brand', 'content']].copy() for df in df_list], ignore_index=True)
# test_df = combined_df.copy()

# # 2. Kiwi 초기화 (num_workers=os.cpu_count() → C++ 워커 스레드 활성화, Jupyter 충돌 없음)
# kiwi = Kiwi(num_workers=os.cpu_count())

# user_words = [
#     '안다르', '휠라', '젝시믹스', '룰루레몬', 
#     '에어리핏', '에어스트', '에어쿨링', '기모', '조거팬츠',
#     '얼라인', '스쿠바', '스위프틀리', '맨즈', '롱슬리브', '숏슬리브',
#     '블랙라벨', '에샤페', '테니스', '착화감', '스탠다드핏', '오버핏', '슬림핏', '크롭티', '고프코어'
# ]
# for word in user_words:
#     kiwi.add_user_word(word, 'NNP')

# def process_result(tokens):
#     return " ".join([
#         t.form + '다' if t.tag == 'VA' else t.form
#         for t in tokens
#         if (t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form == '핏')) or t.tag == 'VA'
#     ])

# print("형태소 분석 시작 (C++ 멀티코어 배치 처리)...")

# # 3. 리스트 일괄 전달 → kiwipiepy 내부 C++ 병렬 배치 처리
# contents = test_df['content'].fillna('').astype(str).tolist()
# test_df['tokens'] = [process_result(r) for r in kiwi.tokenize(contents)]

# print("분석 완료! 시각화를 준비합니다.\n")

# # 4. 시각화
# brand_names = test_df['brand'].unique()

# refined_stopwords = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '구매', '제품', '상품', '주문', '배송', 
#     '이번', '선택', '사용', '아울렛', '로그인', '사이트', '재구매', '이용', '하나', '정도',
#     '좋다', '같다', '있다', '없다', '크다', '작다', '적당하다', '괜찮다', '그냥', '진짜', 
#     '너무', '생각', '느낌', '부분', '사람', '마음', '소리', '보고', '조금', '가장',
#     '사이즈', '색상', '컬러', '화면', '실물', '사진', '재질', '소재', '다르다', '정말',
#     '편하다', '예쁘다', '이쁘다', '편안하다', '만족', '최고', '불편',
#     '평소', '처음', '요즘', '항상', '자주', '안다르', '휠라', '젝시믹스', '룰루레몬',
#     '고민', '구입', '선물', '남편', '추가', '입다', '신다', '많다', '길이',
#     '리뷰', '추천', '세탁', '빠르다', '친구', '매치', '걱정', '무난', '감사', '필요', '치수', '검정',
# }

# for b_name in brand_names:
#     brand_df = test_df[test_df['brand'] == b_name]
#     if brand_df.empty: continue

#     all_words = " ".join(brand_df['tokens']).split()
#     filtered_words = [w for w in all_words if w not in refined_stopwords]
    
#     if not filtered_words: continue
        
#     counts = Counter(filtered_words)
    
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=1000, height=600, max_words=60, colormap='Dark2'
#     ).generate_from_frequencies(counts)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Brand : {b_name}", fontsize=20, fontweight='bold')
#     plt.axis('off')
#     plt.show()

##### 5차 워드클라우드 진행

In [15]:
# # 1. 데이터 합치기
# df_list = [df_andar, df_fila, df_xexymix, df_lululemon]
# combined_df = pd.concat([df[['brand', 'content']].copy() for df in df_list], ignore_index=True)
# test_df = combined_df.copy()

# # 2. Kiwi 초기화 (num_workers=os.cpu_count() → C++ 워커 스레드 활성화, Jupyter 충돌 없음)
# kiwi = Kiwi(num_workers=os.cpu_count())

# user_words = [
#     '안다르', '휠라', '젝시믹스', '룰루레몬', 
#     '에어리핏', '에어스트', '에어쿨링', '기모', '조거팬츠',
#     '얼라인', '스쿠바', '스위프틀리', '맨즈', '롱슬리브', '숏슬리브',
#     '블랙라벨', '에샤페', '테니스', '착화감', '스탠다드핏', '오버핏', '슬림핏', '크롭티', '고프코어'
# ]
# for word in user_words:
#     kiwi.add_user_word(word, 'NNP')

# def process_result(tokens):
#     return " ".join([
#         t.form + '다' if t.tag == 'VA' else t.form
#         for t in tokens
#         if (t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form == '핏')) or t.tag == 'VA'
#     ])

# print("형태소 분석 시작 (C++ 멀티코어 배치 처리)...")

# # 3. 리스트 일괄 전달 → kiwipiepy 내부 C++ 병렬 배치 처리
# contents = test_df['content'].fillna('').astype(str).tolist()
# test_df['tokens'] = [process_result(r) for r in kiwi.tokenize(contents)]

# print("분석 완료! 시각화를 준비합니다.\n")

# # 4. 시각화
# brand_names = test_df['brand'].unique()

# refined_stopwords = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '구매', '제품', '상품', '주문', '배송', 
#     '이번', '선택', '사용', '아울렛', '로그인', '사이트', '재구매', '이용', '하나', '정도',
#     '좋다', '같다', '있다', '없다', '크다', '작다', '적당하다', '괜찮다', '그냥', '진짜', 
#     '너무', '생각', '느낌', '부분', '사람', '마음', '소리', '보고', '조금', '가장',
#     '사이즈', '색상', '컬러', '화면', '실물', '사진', '재질', '소재', '다르다', '정말',
#     '편하다', '예쁘다', '이쁘다', '편안하다', '만족', '최고', '불편', '맨즈', '레깅스',
#     '평소', '처음', '요즘', '항상', '자주', '안다르', '휠라', '젝시믹스', '룰루레몬',
#     '고민', '구입', '선물', '남편', '추가', '입다', '신다', '많다', '길이',
#     '리뷰', '추천', '세탁', '빠르다', '친구', '매치', '걱정', '무난', '감사', '필요', '치수', '검정',
#     '운동', '운동복', '바지', '팬츠', '티셔츠', '상의', '하의', 
#     '디자인', '가격', '색깔', '색감', '착용', '착용감', '원단', '소재', '두께', '신축',
# }

# for b_name in brand_names:
#     brand_df = test_df[test_df['brand'] == b_name]
#     if brand_df.empty: continue

#     all_words = " ".join(brand_df['tokens']).split()
#     filtered_words = [w for w in all_words if w not in refined_stopwords]
    
#     if not filtered_words: continue
        
#     counts = Counter(filtered_words)
    
#     wc = WordCloud(
#         font_path='/System/Library/Fonts/Supplemental/AppleGothic.ttf',
#         background_color='white',
#         width=1000, height=600, max_words=60, colormap='Dark2'
#     ).generate_from_frequencies(counts)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(f"Brand : {b_name}", fontsize=20, fontweight='bold')
#     plt.axis('off')
#     plt.show()

##### 워드클라우드 기반 사용자 사전 초안 작성

##### 샘플데이터 추출 후 원래 리뷰 및 토큰화된 내용 비교 작업 시작

In [16]:
df['content'] = df['content'].apply(clean_text)

In [ ]:

# # ─────────────────────────────────────────────
# # 1. 브랜드별 비율 샘플 추출 (총 500건)
# # ─────────────────────────────────────────────
# SAMPLE_SIZES = {'안다르': 150, '젝시믹스': 150, 'FILA': 100, '룰루레몬': 100}

# test_df = pd.concat([
#     df[df['brand'] == brand].sample(n=n, random_state=42)
#     for brand, n in SAMPLE_SIZES.items()
# ], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# # ─────────────────────────────────────────────
# # 2. Kiwi 초기화
# # ─────────────────────────────────────────────
# kiwi = Kiwi(num_workers=os.cpu_count())

# # ─────────────────────────────────────────────
# # 3. 사용자 사전 (v4.10 업데이트)
# # ─────────────────────────────────────────────
# USER_DICT = [
#     '스탠다드핏', '오버핏', '슬림핏', '테이퍼드핏', '와이드핏', '레귤러핏', '머슬핏', '크롭핏', '루즈핏', '가오리핏', '릴렉스핏',
#     '조거팬츠', '롱슬리브', '숏슬리브', '브라탑', '크롭티', '바람막이', '윈드브레이커', '맨투맨', '아노락', '하프집업', '반집업', '집업', '바이커쇼츠', '와이드팬츠', '트랙팬츠', '자켓', '부츠', '코트',
#     '기모', '심리스', '쿨링', '밴딩', '텐셀', '메쉬', '밑위', '밑단', 'Y존', '넥라인', '암홀', '시보리', '안감', '지퍼', '포켓', '스트링',
#     '신축성', '복원력', '허리말림', '배말림', '압박감', '비침', '땀자국', '땀흡수', '통기성',
#     '보풀', '물빠짐', '이염', '변형', '건조기', '세탁망', '내구성', '톡톡', '탄탄', '실패없다', '필요없다', '마음들다', '후회없다', '설명필요없다',
#     '하비', '하체비만', '상비', '상체비만', '골반', '힙딥', '군살', '체형보정', '비율', '눈바디', '흡습속건', '흉곽압박', '승마살',
#     '갓성비', '가심비', '색맛집', '핏맛집', '재구매', '정사이즈', '반업', '일업', '1업', '깔별', '깔별소장', '데일리템', '휘뚜루마뚜루', '문신템', '문신', '교복템', '교복', '인생바지', '인생레깅스',
#     '웜톤', '쿨톤', '톤다운', '무채색', '파스텔', '고프코어', '발레코어', '블록코어', '볼록코어', '올드머니룩', '와이투케이', 'Y2K', '스트릿룩', '스트릿패션', '캐주얼룩', '캐주얼', '스트릿', '스포티룩', '스포티',
#     '출근룩', '데일리룩', '여행룩', '일상룩', '코디', '셋업', '바프', '바디프로필', '오운완', '원마일웨어', '투마일웨어', '꾸안꾸', '꾸꾸', '꾸꾸꾸',
#     '애슬레저', '에슬레저', '짐웨어', '웰니스', '리커버리', '요가복', '필라테스복', '러닝복', '런닝복', '러닝웨어', '러닝화', '골프웨어', '스윔웨어',
#     '크로스핏', '웨이트', '데드리프트', '스쿼트', '하이킹', '등산', '캠핑', '움직임', '활동성', '슈레이스', '면소재',
#     '휠라', '필라', '휠라코리아', '에샤페', '인터런', '페이토', '타르가', '오크먼트', '레이트레이서', '디스럽터', '디스럽터2', '코트디럭스', '스파게티', '휠라레이', '볼란테', '자마', '실버문', '메탈릭',
#     '휠라키즈', '언더웨어', '헤리티지', '화이트라인', '테니스스커트', '테니스화', '리니어', '밀라노다운', '스포트', '발볼러', '칼발', '찍찍이', '벨크로', '어글리슈즈', '어글리', '청키슈즈', '청키', '빅로고',
#     '안다르', '맨즈', '안다르맨즈', '에어스트', '에어리핏', '올데이핏', '에어쿨링', '뉴에어쿨링', '지니', '비프리', '에어무스', '릴렉스', '서스테이너블', '워터레깅스', '클라우드', '심포니', '에어캐치', '텐셀모달', '코듀라', '샤론', '프라나', '슬랙스', '폴로', '랩가디건', '커버업', '우븐팬츠', '워크레저', '비즈니스캐주얼', '9부', '8.2부', '7부', '임산부레깅스', '임산부', '임부복', 'Y존커버', '에어프라임', '아이스브리드', '노블스트라이프', '썸머라인', '피그먼트',
#     '젝시믹스', '젝시', '젝믹', '젝시맨', '젝시맨즈', '블랙라벨', '블라', '아이스페더', '젤라', '헤라', '텐션', '네오플렉시', '퍼포먼스', '젝시골프', '하이플렉시', '업텐션', '젤라인텐션', '핑거홀', '360N', '380N', '330N', '셀라', '셀라퍼펙션', '쿨파인', '브이업', '아이스페더컴포트', '트루플렉시', '인텐션', '하이서포트', '로우서포트', '미디움서포트', '젝시워터', '젝시코스메틱', '쉐르파',
#     '룰루레몬', '룰루', '얼라인', '스쿠바', '스위프틀리', '원더트레인', '패스트앤프리', '디파인', '메탈벤텍스', '에브리웨어', 'ABC팬츠', '아시아핏', '아시안핏', '글로벌핏', '스웨트라이프', '스윔라이프', '에듀케이터', '눌루', 'Nulu', '에버럭스', 'Everlux', '럭스트림', 'Luxtreme', '소프트스트림', 'Softstreme', '그로브팬츠', '얼라인탱크', '트래커쇼츠', '페이스브레이커', '루온', '실버레센트', '센스니트', '라이선투트레인', '써지', '라이크어클라우드', '에너지브라', '프리투비', '원더퍼프', '댄스스튜디오', '인비고레이트', '베이스페이스', '오티더블유', 'OTW', '차지필', '랩', '얼라인쇼츠',
#     '릴렉스마사지듀얼볼', '듀얼볼', '비즈니스웨어', '데일리웨어', '홈웨어', '이너웨어', '언더레이어', '이너탑', '이너팬츠', '레깅스팬츠', '레깅스', '레그워머', '팔토시', '암워머', '손목밴드', '비즈니스룩', '포멀룩', '오피스룩', '비즈니스캐주얼룩', '요가매트',
#     '빠른배송', '큰사람', '작은사람', '데일리바지', '네온그린', '한사이즈', '반사이즈', '간절기', '쿠션감', '반사이즈업', '반사이즈다운', '한사이즈업', '한사이즈다운', '합리적', '미끄럼방지', '평소', '콤비폴로', '어반액티브', '에어코튼', '스포츠양말', '가격대비', '윈드자켓', '밴드바지', '첫번째', '두번째', '세번째', '바스락', '고객센터', '생산', '쳐박템', '처박템', '아기엄마', '애기엄마', '생활복', '임부용', '기본티', '레이어드티', '레이어드용', '기본템', '데일리웨어바지', '데일리웨어티', '데일리웨어자켓', '데일리신발'
# ]

# for word in set(USER_DICT):
#     kiwi.add_user_word(word, 'NNP')

# # ─────────────────────────────────────────────
# # 4. 정규화 및 불용어 사전 (v4.10)
# # ─────────────────────────────────────────────
# TEXT_CORRECTIONS = {
#     '갠찬': '괜찮', '갠찮': '괜찮', '내요': '네요', '내용': '네요',
#     '실패가 없': '실패없', '실패 없': '실패없', '실패가없': '실패없',
#     '톤톡': '톡톡', '적릭': '적립금', '쵝오': '최고', '오해': '오래',
#     '마음에 들': '마음들다', '맘에 들': '마음들다', '맘에들': '마음들다',
#     '설명이 필요없': '설명필요없다', '설명 필요없': '설명필요없다',
#     '언더': '언더웨어'
# }

# NORMALIZATION_DICT = {
#     '젝시': '젝시믹스', '젝믹': '젝시믹스', '젝시맨': '젝시믹스', '젝시맨즈': '젝시믹스', '룰루': '룰루레몬', '필라': '휠라', '휠라코리아': '휠라', '안다르맨즈': '안다르',
#     '이뻐요': '예쁘다', '이쁘다': '예쁘다', '만적': '만족', '런닝복': '러닝복', '에슬레저': '애슬레저', '블라': '블랙라벨',
#     '조아요': '좋다', '조아여': '좋다', '져아': '좋다', '조아': '좋다', '조으': '좋다',
#     '따뜻해요': '따뜻하다', '따뜻': '따뜻하다', '따듯': '따뜻하다', '캐쥬얼': '캐주얼',
#     '답답': '답답하다', '안답답': '안답답하다', '안불편': '안불편하다', '안비침': '안비치다', '안비쳐': '안비치다',
#     '살아나다': '어울리다', '살아난다': '어울리다', '실패없': '실패없다', '귀엽': '귀엽다', '귀여': '귀엽다', '커엽': '귀엽다', '귀여워': '귀엽다',
#     '고급진': '고급지다', '후회없어요': '후회없다', '마음들다': '마음들다', '추운': '춥다', '고급지': '고급지다', '안부드럽다': '부드럽다'
# }

# RAW_STOPWORDS = {
#     '네이버', '페이', '후기', '작성', '등록', '포인트', '아울렛', '로그인', '사이트', '구매', '제품', '상품', '주문', '배송', '이번', '선택', '사용',
#     '생각', '평소', '사이즈', '사이즈가', '하다', '되다', '들르다', '들렀다', '특유', '햇빛', '조명', '따르다', '시선', '만들다', '측면', '덕분',
#     '신다', '더하다', '필수', '쟁이다', '진짜', '너무', '정도', '많이', '조금', '그냥', '아웃렛', '와이프', '않다', '안다', '내다', '같다', '있다',
#     '울다', '그렇다', '이렇다', '어떻다', '저렇다', '사다', '입다', '순간', '느낌', '착용감', '구입', '종류',
#     '달라다', '넘다', '부분', '알다', '보이다', '중요', '보다', '기준', '올리다', '내리다', '재질', '찾다', '색상', '색깔', '색감', '장난', '죽다', '생명', 
#     '덥다', '힘들다', '삐지다', '받다', '적다', '착용', '살갗', '살갖', '닿다', '타사', '길이', '나오다', '넣다', '방식', '세일', '쎄일', '가격', '안파다', 
#     '배송비', '살다', '할인', '기분', '디다', '약간', '감안', '무릎', '안나오다', '안입다', '요즘', '요즈음', '역쉬', '역시', '기다', '몰다', '모르다', '소재', 
#     '돌리다', '실내', '실외', '바람', '특가', '행사', '가리다', '리뷰', '안타다', '타다', '나가다', '쓰다', '딸아이', '아이', '아들', '남편', '아내', '부인', 
#     '재다', '동생', '언니', '누나', '오빠', '형', '문의', '교환', '감수', '입지', '나다', '오해', '없다', '있다', '치다', '박다', '짙다', '베란다', '일주일', 
#     '두다', '하이', '일반', '비하다', '비슷', '사람', '물어보다', '시즌', '날씨', '껴입다', '복숭아', '딱오다', '도움'
# }
# FINAL_STOPWORDS = RAW_STOPWORDS - set(USER_DICT)

# # ─────────────────────────────────────────────
# # 5. 전처리 함수 (v4.10)
# # ─────────────────────────────────────────────
# def advanced_pre_process(text):
#     for wrong, right in TEXT_CORRECTIONS.items():
#         text = text.replace(wrong, right)
#     text = re.sub(r'([가-힣]+)지[도는]?\s?않\w*', r' 안 \1', text)
#     text = re.sub(r'([가-힣]+)지[도는]?\s?못\w*', r' 못 \1', text)
#     pattern = r'(하고|지만|아서|어서|니까|은데|는데|인데|고요|네요|대요|길래|았는데|었는데|이고)([가-힣])'
#     text = re.sub(pattern, r'\1 \2', text)
#     text = re.sub(r'없어[서선요]', '없다 ', text)
#     text = re.sub(r'필요없\w*', '필요없다', text)
#     return text

# # ─────────────────────────────────────────────
# # 6. 토큰 정제 함수 (v4.10)
# # ─────────────────────────────────────────────
# def process_result(tokens):
#     extracted = []
#     prefix = ""
#     core_single_words = ['핏', '딱', '꽉', '쏙']

#     for t in tokens:
#         if t.tag == 'MAG' and t.form in ['잘', '안', '못', '딱']:
#             prefix = t.form
#             continue

#         is_noun = t.tag in ['NNG', 'NNP', 'XR'] and (len(t.form) > 1 or t.form in core_single_words)
#         is_pred = t.tag in ['VA', 'VA-I', 'VA-R', 'VV', 'VV-I', 'VV-R']

#         if is_noun or is_pred:
#             word = NORMALIZATION_DICT.get(t.form)
#             if word is None:
#                 temp_word = t.form + '다' if is_pred else t.form
#                 word = NORMALIZATION_DICT.get(temp_word, temp_word)

#             if prefix:
#                 word = prefix + word
#                 word = NORMALIZATION_DICT.get(word, word)

#             prefix = ""

#             if word not in FINAL_STOPWORDS:
#                 if len(word) > 1 or word in core_single_words:
#                     extracted.append(word)
#         else:
#             prefix = ""

#     return " ".join(extracted)

# # ─────────────────────────────────────────────
# # 7. 실행
# # ─────────────────────────────────────────────
# print("\n[v4.10] 최종 요청 사항 반영 파이프라인 실행 중...")
# raw_contents = test_df['content'].fillna('').astype(str).tolist()

# processed_texts = [kiwi.space(advanced_pre_process(text)) for text in raw_contents]
# test_df['tokens'] = [process_result(r) for r in kiwi.tokenize(processed_texts)]

# print("✅ 완료! 엑셀 파일 저장 중...")
# inspection_df = test_df[['brand', 'content', 'tokens']]
# inspection_df.to_excel('./step1_token_inspection_v4_10.xlsx', index=False)

# display(inspection_df.head(20))


[v4.10] 최종 요청 사항 반영 파이프라인 실행 중...
✅ 완료! 엑셀 파일 저장 중...


,brand,content,tokens
0,FILA,직접 신어본 휠라 에샤페 실버문은 메탈릭 실버 특유의 은은한 광택이 햇빛과 조명에 ...,휠라 에샤페 실버문 메탈릭 실버 은은 광택 반짝이다 사로잡다 블랙 밑창 대비 세련 ...
1,안다르,면소재이고 밑단이 생각보다는 넓은편이네요 허벅지 얇으신분들은 속이 보일까 조금 조심...,면소재 밑단 넓다 허벅지 얇다 조심 에어리핏 스웻 5부 쇼츠 잘맞다
2,FILA,와이프 225신는데 딱 맞습니다 너무 이쁘내요 신발도 이쁘지만 와이프가요,딱맞다 예쁘다 신발 예쁘다
3,젝시믹스,젝시에서쉐르파만산게 몇개째인지신상나와서또샀어요이번에지퍼와똑딱이 아주좋네요길이감도만족...,젝시믹스 쉐르파 신상 지퍼 좋다 만족 젝시믹스 실패없다 좋다 겨울 젝시믹스 해결
4,안다르,저는 운동복은 안다르에서만 구매해요 제일편하고 종류가 다양해서 너무좋아요 2024 ...,운동복 안다르 편하다 다양 좋다 만족
5,FILA,가을에 신기 좋고 어느바지에도 잘 어울려서 좋습니다,가을 좋다 바지 잘어울리다 좋다
6,FILA,실물이 훨씬 예쁘네요 귀여워요,실물 예쁘다 귀엽다
7,안다르,가격도 참 좋고 색상이 예뻐서 구입 금방 더러워질까봐 고민했는데 오염에 강해서 좋아...,좋다 예쁘다 더러워지다 고민 오염 강하다 좋다 뭉치다 근육 풀다 릴렉스 마사지 듀얼...
8,안다르,평소 라지 입으나 달라붙게 입고 싶어서 미디움 주문했는데 이쁘게 잘 입을것 같아요 ...,평소 달라붙다 예쁘다 잘입다 붙다 싫어하다 추천 에어로피케 숏슬리브 평소
9,룰루레몬,쫀쫀탑 설명이 필요없는 믿고 입는 룰루레몬 이번엔 탑을 주문했어요 제 사이즈는 16...,쫀쫀 설명필요없다 믿다 룰루레몬 쫀쫀 편안 디자인 원하다 디자인 마음들다 빠른배송 ...


##### 형태소 분석 마스터 테이블 생성 및 데이터셋 추출

In [ ]:
# ── 노트북 실행 셀 ────────────────────────────────────
import importlib, sys
sys.path.insert(0, './송원우')

import check_tokens, preprocess
importlib.reload(check_tokens)
importlib.reload(preprocess)

# (1) 검증 — 강도 부사가 tokens에 살아있고 tokens_topic에서는 사라지는지 확인
samples = [
    '진짜 너무 부드럽고 신축성 매우 좋아요',
    '별로 안편해요 사이즈가 조금 큰 느낌',
    '엄청 좋다 강추합니다',
]
toks = preprocess.preprocess_texts(samples)
for s, t in zip(samples, toks):
    print(f'원문: {s}')
    print(f'tokens(ABSA)      : {t}')
    print(f'tokens_topic(BERT): {preprocess._strip_intensity(t)}')
    print()

# (2) 전체 117만건 재실행 (기존 parquet 덮어쓰기, ~3분)
result = preprocess.preprocess_master(
    input_path  = './final_data/master_preprocessed_260430.csv',
    output_dir  = './final_data',
    chunk_size  = 50_000,
)
print(result)

Kiwi 초기화 중...
  사용자 사전 554개 등록 완료. (가중치 강화 12개)
원문: 진짜 너무 부드럽고 신축성 매우 좋아요
tokens(ABSA)      : 진짜 너무 부드럽다 신축성 매우 좋다
tokens_topic(BERT): 부드럽다 신축성 좋다

원문: 별로 안편해요 사이즈가 조금 큰 느낌
tokens(ABSA)      : 별로 편하다 사이즈 조금 크다 느낌
tokens_topic(BERT): 편하다 사이즈 크다 느낌

원문: 엄청 좋다 강추합니다
tokens(ABSA)      : 엄청 좋다 강추
tokens_topic(BERT): 좋다 강추

[preprocess_master] 시작 — 입력: ./final_data/master_preprocessed_260430.csv
  청크 크기: 50,000건 | 출력: ./final_data
  청크   1:    50,000건 누적 | 경과 0.1분
  청크   2:   100,000건 누적 | 경과 0.3분
  청크   3:   150,000건 누적 | 경과 0.4분
  청크   4:   200,000건 누적 | 경과 0.6분
  청크   5:   250,000건 누적 | 경과 0.7분
  청크   6:   300,000건 누적 | 경과 0.9분
  청크   7:   350,000건 누적 | 경과 1.0분
  청크   8:   400,000건 누적 | 경과 1.2분
  청크   9:   450,000건 누적 | 경과 1.4분
  청크  10:   500,000건 누적 | 경과 1.5분
  청크  11:   550,000건 누적 | 경과 1.7분
  청크  12:   600,000건 누적 | 경과 1.8분
  청크  13:   650,000건 누적 | 경과 2.0분
  청크  14:   700,000건 누적 | 경과 2.1분
  청크  15:   750,000건 누적 | 경과 2.2분
  청크  16:   800,000건 누적 | 경과 2.3분
  청크  17:   850,000건 누적 | 경